# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.date_published}, License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's list all available record sets, their `@id`s, and their fields.

> **Note:** According to the Croissant schema, this dataset's recordSets are defined in the main metadata object.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    print(f"Record set: {rs['@id']}")
    if 'field' in rs:
        print('  Fields:')
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"    - {field.get('@id', '')}")
            else:
                print(f"    - {field}")
    else:
        print("  [No fields listed]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

We'll select the first available record set for demonstration, and show first five records and the columns.

In [ ]:
# If record set list is empty, try to infer from manifest
if len(record_sets) == 0:
    raise RuntimeError("No record sets found in the dataset metadata. Please check the schema.")
else:
    record_sets_ids = [rs['@id'] for rs in record_sets]

print(f"Record set IDs available: {record_sets_ids}")

# We'll use the first record set as the main one for demonstration
main_record_set_id = record_sets_ids[0]

# Extract data from all available record sets
dataframes = {}
for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set {rs_id} with columns: {dataframes[rs_id].columns.tolist()}")
    else:
        print(f"No records found for record set {rs_id}.")

# Show first five rows for the main record set
if main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())
else:
    print(f"Main record set {main_record_set_id} does not have any records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll use one numeric field and one categorical field from the main record set for demonstration.

**Please inspect `dataframes[main_record_set_id].columns` output above if you want to adjust the field names!** For demonstration purposes, we'll try sensible default field names.

In [ ]:
# Choose a numeric and group (categorical) field
df = dataframes[main_record_set_id]

# Attempt to auto-detect numeric and categorical columns
numeric_field = None
group_field = None
# Try common candidates
candidates = ['coverage_percent', 'peptide_count', 'MW', 'pI', 'abundance', 'MolecularWeight_kDa', 'Coverage_%', 'UniquePeptides', 'TotalPeptides']
for col in df.columns:
    if numeric_field is None and any(cand.lower() in col.lower() for cand in candidates):
        numeric_field = col
# Try to find a group field (e.g. description or accession or sample)
for col in df.columns:
    if group_field is None and ('description' in col.lower() or 'accession' in col.lower() or 'sample' in col.lower() or 'group' in col.lower()):
        group_field = col

print(f"Chosen numeric field: {numeric_field}")
print(f"Group/categorical field: {group_field}")

if numeric_field is not None:
    # Try converting to float for numeric operations
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(12, 6))
        top_groups = df[group_field].value_counts().head(10).index
        sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field} (top 10 groups)')
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("Skipping visualization: No numeric field detected.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
* Loaded Croissant-formatted metadata and data using `mlcroissant`;
* Explored record sets and fields using the Croissant `@id` mechanism;
* Loaded records into pandas DataFrames for analysis;
* Performed exploratory data analysis targeting numeric fields;
* Visualized distributions and group differences.

This approach can be extended to more in-depth analysis, machine learning, or data integration workflows, leveraging the FAIR structure provided.